# End-to-End Pipeline — Sprints 2 and 3

This notebook loads the prepared GitSkills DuckDB database and calls `summarize_cohort()` using the family names in the selected `samples/families.txt` file. Choose **Sprint 2** or **Sprint 3, sample 1–3** in the configuration cell. All pipeline-generated artifacts—including `sample_manifest.json`, validation samples, manual-review files, audit records, summaries, and figures—are saved to the matching `results` directory.

Compatible scan files in the shared `.scans/` directory alongside the notebook are reused across cohorts; stale or missing scans are regenerated. Full similarity evidence is held in memory only while selecting manual-review examples.


### Project Imports

In [ ]:
import json
import os
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import SVG, display

from gitskills.analysis.family_models import SimilarityPolicy
from gitskills.pipelines.cohort import summarize_cohort
from gitskills.tools._family_input import read_family_names


### Set Variables

In [ ]:
DEFAULT_DB_PATH = Path(r"C:\data\duckdb\agent_skills_release.db")
DB_PATH = Path(
    os.environ.get("GITSKILLS_DB", DEFAULT_DB_PATH)
)
if not DB_PATH.is_file():
    raise FileNotFoundError(f"Prepared DuckDB not found: {DB_PATH}. Set GITSKILLS_DB.")

PROJECT_ROOT = next(
    (directory for directory in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (directory / "pyproject.toml").is_file() and (directory / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from the project repository (or a directory beneath it).")

NOTEBOOK_DIR = PROJECT_ROOT / "notebooks" / "end_to_end_pipeline"
SPRINT = 2  # 2 for the original cohort; 3 for one of three new cohorts.
SAMPLE = 1  # For Sprint 3: 1, 2, or 3 (ignored for Sprint 2).

if SPRINT == 2:
    SAMPLE_DIR = NOTEBOOK_DIR / "sprint2" / "samples"
    RESULT_DIR = NOTEBOOK_DIR / "sprint2" / "results"
elif SPRINT == 3 and SAMPLE in (1, 2, 3):
    SAMPLE_DIR = NOTEBOOK_DIR / "sprint3" / f"samples{SAMPLE}"
    RESULT_DIR = NOTEBOOK_DIR / "sprint3" / f"results{SAMPLE}"
else:
    raise ValueError("Set SPRINT=2 or SPRINT=3 with SAMPLE=1, 2, or 3")

FAMILY_FILE = SAMPLE_DIR / "families.txt"
SCAN_CACHE_DIR = NOTEBOOK_DIR / ".scans"


### Locate Database

In [ ]:
print(f"Database: {DB_PATH}")
print(f"Size: {DB_PATH.stat().st_size / 1024**3:.2f} GB")

### Connect to DuckDB

In [ ]:
with duckdb.connect(database=str(DB_PATH), read_only=True) as connection:
    duckdb_version = connection.execute("SELECT version()").fetchone()[0]
    available_tables = {row[0] for row in connection.execute("SHOW TABLES").fetchall()}
    required_tables = {"artifacts", "artifact_groupings", "artifact_siblings", "repos"}
    if missing := required_tables - available_tables:
        raise RuntimeError(f"Prepare the analysis database first; missing tables: {sorted(missing)}")

print(f"DuckDB {duckdb_version} | {DB_PATH}")
print(f"Family selection: {FAMILY_FILE.relative_to(PROJECT_ROOT)}")
print(f"Results: {RESULT_DIR.relative_to(PROJECT_ROOT)}")
print(f"Scan cache: {SCAN_CACHE_DIR.relative_to(PROJECT_ROOT)}")


### Select candidate families

Read the exact family names from `families.txt` (one name per line; empty lines and `#` comments are ignored). Sprint 2 uses `sprint2/samples/families.txt`; Sprint 3 uses `sprint3/samples1`, `samples2`, or `samples3`, selected above. Generate the Sprint 3 files with `family_picker` before running the notebook.

Set `APPROVAL_REFERENCE` to the team decision/date after approval; the manifest otherwise marks the cohort provisional. All directed edges (primary and additional) are included; ambiguous relationships remain eligible only for relationship validation.


In [ ]:
if not FAMILY_FILE.is_file():
    raise FileNotFoundError(f"Family list not found: {FAMILY_FILE}. Provide families.txt before running.")
candidate_families = read_family_names(FAMILY_FILE)

POLICY = SimilarityPolicy()  # Shared similarity/direction thresholds; see RDR-004 and RDR-007.
EDGES = "all"  # Both primary and additional direction-supported comparisons.
PER_STRATUM = 5
SEED = 580
APPROVAL_REFERENCE = None  # Set to a recorded team decision/date after approval.

print(f"Selected families: {len(candidate_families)}")


### Summarize the selected cohort

The summarizer loads each selected family from DuckDB, runs the family analyzer for relationship validation, and reuses compatible scan reports in `SCAN_CACHE_DIR`. It creates or refreshes missing/stale scans and writes **cohort-specific artifacts** to `RESULT_DIR`, including the cohort manifest, manual-review samples and templates, research tables, figures, and audit records. Use `rescan=True` to force all scans to be rebuilt.


In [ ]:
execution = summarize_cohort(
    candidate_families,
    database=DB_PATH,
    results=RESULT_DIR,
    scan_cache=SCAN_CACHE_DIR,
    policy=POLICY,
    edges=EDGES,
    per_stratum=PER_STRATUM,
    seed=SEED,
    approval_reference=APPROVAL_REFERENCE,
)

display(pd.DataFrame([execution]))
print(f"Research summary: {RESULT_DIR / 'review' / 'summary.json'}")
print(f"Sample manifest: {RESULT_DIR / 'sample_manifest.json'}")
print(f"Manual review: {RESULT_DIR / 'manual-review'}")
print(f"Processing audit: {RESULT_DIR / 'audit' / 'processing_summary.csv'}")


### Check research outputs

The shared scan cache resides in `end_to_end_pipeline/.scans/`, outside any sprint results directory and independently of SkillTrace's compact lineage reports. The following section confirms which families were processed and the number of directed comparisons in each report.

In [ ]:
runs = pd.read_csv(RESULT_DIR / "audit" / "processing_summary.csv")
display(runs[["family", "artifacts_loaded", "artifacts_skipped", "pairs_analyzed", "pairs_incomplete"]])

### Preliminary results: directed comparisons

Counts describe **direction-supported comparisons**, not independent security incidents or distinct artifacts. A *flagged* pair is a broader category than a pair with a scanner-supported risk indicator; manual validation is still pending.

In [ ]:
REVIEW_DIR = RESULT_DIR / "review"
FIGURES_DIR = RESULT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

with (REVIEW_DIR / "summary.json").open(encoding="utf-8") as file:
    summary = json.load(file)

total = summary["directed_pairs_analyzed"]
flagged = summary["flagged_pairs"]
indicated = summary["pairs_with_supported_indicators"]
if not 0 <= indicated <= flagged <= total:
    raise ValueError("Inconsistent directed-pair counts in summary.json")

outcomes = pd.Series({
    "Not flagged": total - flagged,
    "Flagged without supported indicator": flagged - indicated,
    "With supported indicator": indicated,
})
display(pd.DataFrame({
    "Pairs": outcomes,
    "Percent": (100 * outcomes / total).round(1) if total else 0.0,
}))
print(f"{summary['supported_findings']} individual indicator findings across {indicated} pairs")

if total:
    fig, ax = plt.subplots(figsize=(9, 3))
    bars = ax.barh(outcomes.index, outcomes.values)
    ax.bar_label(bars, labels=[f"{count} ({count / total:.1%})" for count in outcomes], padding=4)
    ax.set_xlim(0, total * 1.3)
    ax.set_xlabel("Directed comparisons")
    ax.invert_yaxis()
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "pair_outcomes.svg", bbox_inches="tight")
    plt.show()
else:
    print("No direction-supported comparisons in this cohort.")

### Results by candidate family

Compare the number of analyzed pairs with the number containing indicators. Rates use each family's **own** pair count; families without directed comparisons have no rate. Sprint 2 uses an intentionally selected cohort; Sprint 3 can analyze three independently selected family-picker cohorts, subject to the picker's sampling rules.


In [ ]:
families = pd.read_csv(REVIEW_DIR / "families.csv")
if set(families["family"]) != set(candidate_families):
    raise ValueError("families.csv does not match the selected candidate_families")

families["indicator_rate_pct"] = (
    100 * families["pairs_with_indicators"]
    / families["pairs_analyzed"].replace(0, float("nan"))
)
display(families[[
    "family", "artifacts_loaded", "pairs_analyzed", "pairs_flagged",
    "pairs_with_indicators", "indicator_rate_pct",
]].sort_values("pairs_analyzed", ascending=False).round({"indicator_rate_pct": 1}))

comparable = families.loc[families["pairs_analyzed"] > 0].sort_values("indicator_rate_pct")
if not comparable.empty:
    fig, ax = plt.subplots(figsize=(10, max(3, len(comparable) * 0.45 + 1)))
    bars = ax.barh(comparable["family"], comparable["indicator_rate_pct"])
    for bar, (_, row) in zip(bars, comparable.iterrows()):
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                f"{int(row['pairs_with_indicators'])}/{int(row['pairs_analyzed'])}",
                va="center", fontsize=9)
    ax.set_xlim(0, 115)
    ax.set_xlabel("Pairs with supported indicators (%)")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "indicators_by_family.svg", bbox_inches="tight")
    plt.show()

no_pairs = families.loc[families["pairs_analyzed"] == 0, "family"].tolist()
if no_pairs:
    print(f"No direction-supported comparisons: {', '.join(no_pairs)}")
if total:
    largest = families.sort_values("pairs_analyzed", ascending=False).iloc[0]
    if largest["pairs_analyzed"] > total / 2:
        print(f"Cohort imbalance: {largest['family']} contributes "
              f"{largest['pairs_analyzed'] / total:.1%} of directed comparisons.")

### Risk indicators detected

Display the figure already generated by `summarize_cohort()` and the observed indicator types. Counts are **pairs containing an indicator**, not finding occurrences; one pair may contain multiple indicator types.

In [ ]:
display(SVG(filename=str(FIGURES_DIR / "indicator_frequency.svg")))

indicator_labels = {
    "RI-001": "Download-to-interpreter execution",
    "RI-002": "Remote execution source changed",
    "RI-003": "New network destination or outbound transfer",
    "RI-004": "Weakened network verification/transport",
    "RI-005": "External package or script acquisition",
    "RI-006": "Destructive or privileged file operation",
    "RI-007": "Credential access or outbound reference",
    "RI-008": "System or permissions modification",
    "RI-009": "Sensitive file operation",
    "RI-010": "Local script invocation or content change",
    "RI-011": "External instruction dependency change",
}
frequency = pd.read_csv(REVIEW_DIR / "indicator_frequency.csv")
observed = frequency.loc[frequency["pairs"] > 0].copy()
observed["Description"] = observed["indicator_id"].map(indicator_labels).fillna("See scanner report")
display(observed[["indicator_id", "Description", "pairs", "percent_of_directed_pairs"]]
        .sort_values("pairs", ascending=False).rename(columns={
            "indicator_id": "Indicator", "pairs": "Pairs",
            "percent_of_directed_pairs": "Percent of directed pairs",
        }).round(1))
if observed.empty:
    print("No supported risk indicators in the selected cohort.")

### Edge types and analysis coverage

Primary and additional edges can share source artifacts; their rates are descriptive, not independent risk estimates. Ambiguous relationships are excluded from directed scanning but remain eligible for relationship validation.

In [ ]:
edges = pd.read_csv(REVIEW_DIR / "edge_types.csv")
edges["indicator_rate_pct"] = (
    100 * edges["pairs_with_indicators"]
    / edges["pairs_analyzed"].replace(0, float("nan"))
)
display(edges[[
    "edge_type", "pairs_analyzed", "pairs_flagged", "pairs_with_indicators",
    "indicator_rate_pct", "pairs_coverage_incomplete",
]].round({"indicator_rate_pct": 1}))

edge_plot = edges.loc[edges["pairs_analyzed"] > 0]
if not edge_plot.empty:
    fig, ax = plt.subplots(figsize=(7, 2.6))
    bars = ax.barh(edge_plot["edge_type"], edge_plot["indicator_rate_pct"])
    for bar, (_, row) in zip(bars, edge_plot.iterrows()):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                f"{int(row['pairs_with_indicators'])}/{int(row['pairs_analyzed'])}",
                va="center")
    ax.set_xlim(0, max(5, edge_plot["indicator_rate_pct"].max() * 1.35))
    ax.set_xlabel("Pairs with supported indicators (%)")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "indicators_by_edge_type.svg", bbox_inches="tight")
    plt.show()

with (RESULT_DIR / "execution_summary.json").open(encoding="utf-8") as file:
    execution_summary = json.load(file)
coverage = pd.Series({
    "Artifacts loaded": execution_summary["artifacts_loaded"],
    "Artifacts skipped": execution_summary["artifacts_skipped"],
    "Pairs with changed evidence": summary["pairs_with_evidence_changes"],
    "Pairs with changed sibling evidence": summary["pairs_with_sibling_evidence_changes"],
    "Pairs with incomplete evidence coverage": summary["pairs_with_incomplete_evidence_coverage"],
    "Ambiguous edges not direction-scanned": summary["ambiguous_edges_skipped"],
    "Directed review samples": summary["sampling"]["sampled_pairs"],
    "Relationship review samples": summary["relationship_sampling"]["sampled_pairs"],
})
display(coverage.rename("Count").to_frame())
print(f"Validation: {execution_summary['validation_status']}")

### Findings for manual review

Each row is a directed artifact comparison containing at least one scanner-supported indicator. Use its source IDs to locate the original artifacts in `RESULT_DIR/sample_manifest.json` or SkillTrace. Incomplete coverage requires particular care; these are **review candidates**, not confirmed vulnerabilities.


In [ ]:
directed = pd.read_csv(REVIEW_DIR / "directed_pairs.csv")
if len(directed) != total:
    raise ValueError("directed_pairs.csv does not match summary.json")

findings = directed.loc[directed["indicator_ids"].fillna("").str.strip().ne("")].copy()
display(findings[[
    "family", "base_artifact_id", "derived_artifact_id", "edge_type",
    "direction_basis", "indicator_ids", "finding_count", "coverage_status",
]].sort_values(["family", "base_artifact_id", "derived_artifact_id"]).reset_index(drop=True))
print(f"{len(findings)} pairs to inspect; "
      f"{(findings['coverage_status'] != 'complete').sum()} with incomplete coverage.")

### Independent review and adjudication (team action)

The pipeline is complete when the outputs above are written; **human validation is separate**. Follow [`docs/instructions/manual_annotation_protocol.md`](../../docs/instructions/manual_annotation_protocol.md):

1. Give each reviewer an independent copy of `RESULT_DIR/manual-review/directed-pairs/annotations_template.csv` and `RESULT_DIR/manual-review/relationship-pairs/relationship_annotations_template.csv`.
2. Inspect source artifacts and siblings using the IDs and paths in `RESULT_DIR/sample_manifest.json` and SkillTrace. Do not run stored scripts.
3. Preserve individual reviews and resolve disagreements in separate adjudicated CSVs (recommended: `RESULT_DIR/manual-review/adjudicated/`). Do not overwrite generated templates.
4. Once both reviews are adjudicated, run the optional cell below. It verifies the saved cohort and validation samples, then updates `RESULT_DIR/review/summary.json` without rerunning family similarity analysis or security scans.

If the cohort, policy, or rules change after review begins, archive the old results and validation copies before creating a new sample.


In [ ]:
directed_review = RESULT_DIR / "manual-review" / "adjudicated" / "directed.csv"
relationship_review = RESULT_DIR / "manual-review" / "adjudicated" / "relationships.csv"

if directed_review.is_file() and relationship_review.is_file():
    completed = summarize_cohort(
        candidate_families, database=DB_PATH, results=RESULT_DIR, scan_cache=SCAN_CACHE_DIR, policy=POLICY, edges=EDGES, per_stratum=PER_STRATUM, seed=SEED,
        approval_reference=APPROVAL_REFERENCE,
        directed_annotations=directed_review,
        relationship_annotations=relationship_review,
    )
    print("Adjudicated sample metrics written to review/summary.json")
else:
    print("Manual review is pending. See manual-review/ and copy the templates for reviewers.")